In [ ]:
%load_ext aiida
%aiida
%verdi status

In [2]:
import sys
sys.path.insert(0, '/path/to/aiida')
import copy
import ase.io
from aiida import orm, engine
from ase.build import bulk
import MOFSCO_WorkChain_scf_PBE_final_kraken as MOFSCO_WorkChain_scf
from aiida.engine import run, WorkChain
from aiida.orm import Dict, KpointsData, StructureData, load_code, load_group, Float
from aiida_quantumespresso.workflows.pw.relax import PwRelaxWorkChain
from aiida_quantumespresso.workflows.pw.base import PwBaseWorkChain
import aiida_quantumespresso

In [3]:
pseudo_family = load_group('PseudoDojo/0.4/PBE/SR/standard/upf')
code = load_code('code_name')

In [4]:
import json
import numpy as np
import pandas as pd

#load file with total magnetization computed for all MOFs using oxidation states
f = open('Total_magnetization_2184.json')
data = json.load(f)
data_sorted = sorted(data, key=lambda d: d['n_atoms'])

#list of MOF names
test_set_qmof_id_path = 'selected_mofs.csv'
test_set_qmof_id = pd.read_csv(test_set_qmof_id_path)['MOF_Name'].to_numpy()
test_set_qmof_id.sort()

#saving LS and HS total magentization for the test set into arrays
j=0
test_set = []
for i in range(len(data_sorted)):
    if data_sorted[i]['MOF'] in test_set_qmof_id:
        test_set = np.append(test_set,data_sorted[i])
        print(j,data_sorted[i]['MOF'],data_sorted[i]['n_atoms'],data_sorted[i]['total_M_LS'],
              data_sorted[i]['total_M_HS'],data_sorted[i]['metal'])
        j+=1
print(len(test_set))

0 qmof-015ecfe 27 1 3 Co
1 qmof-852524d 27 1 3 Co
2 qmof-b808abd 31 0 2 Ni
3 qmof-3a8fabc 32 2 6 Co
4 qmof-dbedd04 34 2 6 Co
5 qmof-f07f203 35 1 3 Co
6 qmof-849cda5 37 1 3 Co
7 qmof-8aa3a8e 37 1 3 Co
8 qmof-687e4d6 41 1 3 Co
9 qmof-4ee0d95 42 2 6 Co
10 qmof-cc4910d 45 1 3 Co
11 qmof-ce36f66 45 1 3 Co
12 qmof-41cbe0b 47 1 3 Co
13 qmof-1c152b7 49 1 3 Co
14 qmof-fd921b0 50 2 6 Co
15 qmof-4f2905b 53 1 3 Co
16 qmof-7787227 58 2 6 Co
17 qmof-ab5d6dc 58 2 6 Co
18 qmof-f47b0d0 58 2 6 Co
19 qmof-923ce21 59 1 3 Co
20 qmof-2e6cac6 61 1 3 Co
21 qmof-458a962 61 1 3 Co
22 qmof-110e680 62 2 6 Co
23 qmof-f1d2d60 63 3 9 Co
24 qmof-d8703ab 66 2 6 Co
25 qmof-f260694 66 2 6 Co
26 qmof-2de35b5 67 1 3 Co
27 qmof-f73a75b 67 0 4 Fe
28 qmof-38220b3 70 2 6 Co
29 qmof-623345d 71 1 3 Co
30 qmof-bb2839a 71 1 3 Co
31 qmof-8fe6d7b 72 2 6 Co
32 qmof-053334d 73 1 3 Co
33 qmof-29090e7 73 1 3 Co
34 qmof-547262c 75 1 3 Co
35 qmof-94a67e3 75 1 3 Co
36 qmof-ac415a0 75 1 3 Co
37 qmof-2a2a9c6 78 2 6 Co
38 qmof-7bea0fe 78 2 6

## Run the workchain

In [6]:
for i in range(2, 10):  # change range to run more MOFs
    
    mof = test_set[i]['MOF']
    ls_total_magnetization = test_set[i]['total_M_LS']
    hs_total_magnetization = test_set[i]['total_M_HS']
    is_total_magnetization = test_set[i]['total_M_IS']
    metal = test_set[i]['metal']
    
    ase_structure = ase.io.read("sco_2184/" + mof + '.cif')
    aiida_structure = StructureData(ase=ase_structure)
    aiida_structure.store()
    print(i, mof, ls_total_magnetization, hs_total_magnetization, is_total_magnetization, test_set[i]['n_atoms'], metal)

    cutoff_wfc, cutoff_rho = pseudo_family.get_recommended_cutoffs(structure=aiida_structure, unit='Ry')

    # SCF parameter list
    parameters_scf = Dict({
        'CONTROL': {'calculation': 'scf'},
        'SYSTEM': {
            'ecutwfc': cutoff_wfc,
            'ecutrho': cutoff_rho,
            'nspin': 2,
            'degauss': 0.01,
            'occupations': 'smearing',
            'tot_magnetization': ls_total_magnetization,
            'vdw_corr': 'DFT-D3',
            'dftd3_version': 4,
        },
        'ELECTRONS': {
            'mixing_beta': 0.2,
            'conv_thr': 1e-6,
            'electron_maxstep': 400,
        }
    })

    print('Running 2 scfs \n')
    engine.submit(MOFSCO_WorkChain_scf_PBE_final.MOFSCO_Workchain,
                      structure=aiida_structure,
                      params_scf=parameters_scf,
                      code=code,
                      pseudo_family=orm.Str('PseudoDojo/0.4/PBE/SR/standard/upf'),
                      group=orm.Str('prediction_set_PBE'),
                      mof=orm.Str(mof),
                      hs_total_magnetization=orm.Float(hs_total_magnetization),
                      num_machines=orm.Int(1),
                      max_iterations=orm.Int(5),
                      walltime_scf=orm.Int(43200))

2 qmof-b808abd 0 2 0 31 Ni
Running 2 scfs 

3 qmof-3a8fabc 2 6 0 32 Co
Running 2 scfs 

4 qmof-dbedd04 2 6 0 34 Co
Running 2 scfs 

5 qmof-f07f203 1 3 0 35 Co
Running 2 scfs 

6 qmof-849cda5 1 3 0 37 Co
Running 2 scfs 

7 qmof-8aa3a8e 1 3 0 37 Co
Running 2 scfs 

8 qmof-687e4d6 1 3 0 41 Co
Running 2 scfs 

9 qmof-4ee0d95 2 6 0 42 Co
Running 2 scfs 



## Extract results

In [9]:
group = load_group('prediction_set_PBE')
deltaE = []
for i in range(2,5):
    
    mof = test_set[i]['MOF']
    nodes = [node for node in group.nodes if node.base.extras.all['MOF']==mof]
    print(i)
    ls_energy = hs_energy = is_energy = 0
    for node in nodes:
        
        if node.base.attributes.all['process_state']=='finished':
            
            print(node.base.attributes.all['exit_status'], node.pk,test_set[i]['MOF'],
                 node.base.extras.all['MOF'],node.base.extras.all['spin_state'])
            
            if node.base.attributes.all['exit_status']==0:

                if node.base.extras.all['spin_state']=='LS' and node.base.extras.all['calculation']=='scf':
                    ls_energy = node.outputs.output_parameters['energy'] 
                    #print('LS energy = ',ls_energy, node.outputs.output_parameters['total_magnetization'] )
                elif node.base.extras.all['spin_state']=='HS' and node.base.extras.all['calculation']=='scf':
                    hs_energy = node.outputs.output_parameters['energy']
                    #print('HS energy = ',hs_energy, node.outputs.output_parameters['total_magnetization'] )
                elif node.base.extras.all['spin_state']=='IS' and node.base.extras.all['calculation']=='scf':
                    is_energy = node.outputs.output_parameters['energy']
                    #print('IS energy = ',is_energy, node.outputs.output_parameters['total_magnetization'] )
        if ls_energy != 0 and hs_energy != 0:
            print('DeltaE (H-L) = ', hs_energy - ls_energy)
    print('************************************************************************************************************************* \n')

2
0 424 qmof-b808abd qmof-b808abd LS
0 438 qmof-b808abd qmof-b808abd HS
DeltaE (H-L) =  0.9988659180999093
************************************************************************************************************************* 

3
0 463 qmof-3a8fabc qmof-3a8fabc LS
0 474 qmof-3a8fabc qmof-3a8fabc HS
DeltaE (H-L) =  -0.6827803240012145
************************************************************************************************************************* 

4
0 509 qmof-dbedd04 qmof-dbedd04 LS
0 513 qmof-dbedd04 qmof-dbedd04 HS
DeltaE (H-L) =  0.9926665660004801
************************************************************************************************************************* 

